In [2]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types


In [4]:
import sqlite3

def setup_mock_database():
    conn = sqlite3.connect("company.db")
    cursor = conn.cursor()
    
    # Create clean structured tables
    cursor.execute("DROP TABLE IF EXISTS orders")
    cursor.execute("DROP TABLE IF EXISTS products")
    
    cursor.execute("""
        CREATE TABLE products (
            product_id INTEGER PRIMARY KEY,
            name TEXT,
            price REAL
        )
    """)
    cursor.execute("""
        CREATE TABLE orders (
            order_id INTEGER PRIMARY KEY,
            customer_email TEXT,
            product_id INTEGER,
            status TEXT,
            FOREIGN KEY(product_id) REFERENCES products(product_id)
        )
    """)
    
    # Seed real row data
    cursor.executemany("INSERT INTO products VALUES (?, ?, ?)", [
        (1, "Premium Mechanical Keyboard", 129.99),
        (2, "Ergonomic Wireless Mouse", 59.99),
        (3, "4K Ultra-Wide Monitor", 449.99)
    ])
    cursor.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", [
        (101, "alice@email.com", 1, "Shipped"),
        (102, "bob@email.com", 3, "Processing"),
        (103, "charlie@email.com", 2, "Delivered"),
        (104, "alice@email.com", 3, "Shipped")
    ])
    
    conn.commit()
    conn.close()

setup_mock_database()
print("📁 Database created and seeded perfectly!")

📁 Database created and seeded perfectly!


Metadata

In [3]:
database_metadata = """
You are a database admin assistant. You have access to a SQL database.
Here is the exact schema of the database you are allowed to query:

Table 'products':
  - product_id (INTEGER, PRIMARY KEY)
  - name (TEXT) -> The description of the item
  - price (REAL) -> The cost of the item

Table 'orders':
  - order_id (INTEGER, PRIMARY KEY)
  - customer_email (TEXT) -> The email of the person who bought it
  - product_id (INTEGER) -> Foreign key linking to products table
  - status (TEXT) -> Can be 'Shipped', 'Processing', or 'Delivered'

Strict Rule: Only use column and table names that exist above. Do not guess.
"""

In [5]:
def run_sql_query(query_statement: str)->str:
    """Execute Read-Only query into the internal database of the company
       containing tables 'products' and 'orders' to find product and order metrics
        
       Args:
            query_statement: A valid SQLlite string (e.g SELECT * FROM order WHERE order_id=103)

          """
    try:
        conn=sqlite3.connect("company.db")
        cursor=conn.cursor()
        cursor.execute(query_statement)
        result=cursor.fetchall()
        cursor.close()
        return result
    except Exception as e:
        return f"Error happen when executing the query :{str(e)}"